# 🚀 Namma Space – 3D Reconstruction Pipeline
**Techfest IIT Bombay 2026-27 | Round 1**

This notebook converts a smartphone indoor walkthrough video into an optimized `.glb` 3D mesh.

**Pipeline:** OpenCV Frame Extraction → COLMAP SfM (GPU) → PatchMatch Stereo → Screened Poisson Meshing → GLB Export

**Requirements:** Google Colab with T4 GPU runtime.

In [ ]:
#@title 1️⃣ Environment Setup & Video Upload
import os
import subprocess

print('📦 Installing system dependencies (FFmpeg, Xvfb, OpenGL libraries)...')
os.system('apt-get update -qq && apt-get install -y -qq xvfb ffmpeg libglew-dev libqt5opengl5-dev libfreeimage-dev > /dev/null 2>&1')

print('🚀 Installing CUDA-accelerated COLMAP (GPU SIFT + PatchMatch Stereo)...')
gpu_url = 'https://github.com/lyehe/build_gpu_colmap/releases/download/v4.2.1/COLMAP-4.2.1-ubuntu-22.04-CUDA-Caspar.zip'
extract_cmd = (
    f'wget -q -O /tmp/colmap_gpu.zip {gpu_url} && '
    'rm -rf /tmp/colmap_pkg && '
    'unzip -q -o /tmp/colmap_gpu.zip -d /tmp/colmap_pkg && '
    'find /tmp/colmap_pkg -type f -name colmap -exec cp {} /usr/local/bin/ \\; && '
    'chmod +x /usr/local/bin/colmap && '
    'find /tmp/colmap_pkg -mindepth 2 -type d -name lib -exec cp -r {}/* /usr/local/lib/ \\; 2>/dev/null && '
    'ldconfig && '
    'rm -rf /tmp/colmap_gpu.zip /tmp/colmap_pkg'
)
ret = os.system(extract_cmd)

if ret != 0 or os.system('colmap --help > /dev/null 2>&1') != 0:
    print('ℹ️ Note: Using standard package manager fallback...')
    os.system('apt-get install -y -qq colmap > /dev/null 2>&1')
else:
    print('✅ GPU-accelerated COLMAP installed successfully!')

print('📦 Installing Python libraries (PyMeshLab, Trimesh, OpenCV)...')
os.system('pip install -q opencv-python-headless pymeshlab trimesh[easy] numpy')

# Verify GPU
print('\n🖥️ GPU Status:')
os.system('nvidia-smi --query-gpu=name,memory.total --format=csv,noheader')

# Create workspace
os.makedirs('workspace/images', exist_ok=True)
os.makedirs('workspace/sparse', exist_ok=True)
os.makedirs('workspace/dense', exist_ok=True)
print('\n📁 Workspace initialized: workspace/')

# Upload video
from google.colab import files
print('\n📹 Upload your walkthrough video (MP4):')
uploaded = files.upload()
video_path = list(uploaded.keys())[0]
print(f'✅ Video ready: {video_path}')


## 2. Sharp Frame Extraction
Samples frames at 2 FPS and uses OpenCV's Laplacian variance to discard motion-blurred frames.

In [ ]:
#@title 2️⃣ OpenCV Laplacian Sharp-Frame Extractor
import cv2
import numpy as np

cap = cv2.VideoCapture(video_path)
fps = cap.get(cv2.CAP_PROP_FPS)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
sample_interval = max(1, int(fps / 3))  # Sample at ~3 FPS

BLUR_THRESHOLD = 35.0  # Optimal for indoor room lighting (set 35-60)
MAX_WIDTH = 1280

print(f"📹 Video: {video_path}")
print(f"   FPS: {fps:.1f} | Total Frames: {total_frames} | Duration: {total_frames/fps:.1f}s")
print(f"   Sampling every {sample_interval} frames (~2 FPS)")
print(f"   Blur threshold: {BLUR_THRESHOLD}")
print()

count = 0
saved = 0
rejected = 0

while True:
    ret, frame = cap.read()
    if not ret:
        break
    if count % sample_interval == 0:
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        laplacian_var = cv2.Laplacian(gray, cv2.CV_64F).var()
        if laplacian_var >= BLUR_THRESHOLD:
            h, w = frame.shape[:2]
            if w > MAX_WIDTH:
                scale = MAX_WIDTH / w
                frame = cv2.resize(frame, (MAX_WIDTH, int(h * scale)), interpolation=cv2.INTER_AREA)
            cv2.imwrite(f'workspace/images/frame_{saved:05d}.jpg', frame, [cv2.IMWRITE_JPEG_QUALITY, 95])
            saved += 1
        else:
            rejected += 1
    count += 1

cap.release()
print(f"✅ Extracted {saved} sharp frames (rejected {rejected} blurry frames)")
print(f"   Frames saved to: workspace/images/")

# Show sample
import glob
sample_files = sorted(glob.glob('workspace/images/*.jpg'))[:4]
if sample_files:
    from IPython.display import display, Image as IPImage
    print(f"\n📸 Sample extracted frames:")
    for f in sample_files:
        display(IPImage(filename=f, width=400))


## 3. GPU COLMAP Reconstruction + Screened Poisson Meshing
Runs the full COLMAP SfM + MVS pipeline on GPU, then reconstructs a surface mesh using Screened Poisson and decimates to 150K faces.

In [ ]:
#@title 3️⃣ GPU COLMAP SfM + Dense Reconstruction + Meshing
import os
import time
import subprocess

start = time.time()

DB = "workspace/database.db"
IMG = "workspace/images"
SPARSE = "workspace/sparse"
DENSE = "workspace/dense"

os.makedirs(SPARSE, exist_ok=True)
os.makedirs(DENSE, exist_ok=True)

# Remove old database if re-running
if os.path.exists(DB):
    os.remove(DB)

def run_cmd(name, cmd, fallback_cmd=None):
    print(f"\n{'='*55}\n  ▶ {name}\n{'='*55}")
    ret = os.system(cmd)
    if ret != 0:
        if fallback_cmd:
            print(f"⚠️ GPU mode returned code {ret}. Automatically switching to CPU fallback...")
            ret = os.system(fallback_cmd)
        if ret != 0:
            print(f"❌ {name} failed with code {ret}")
            return False
    print(f"✅ {name} complete")
    return True

# 1. Feature extraction: GPU SIFT with auto CPU fallback
cmd_gpu_sift = f"colmap feature_extractor --database_path {DB} --image_path {IMG} --ImageReader.single_camera 1 --SiftExtraction.use_gpu 1"
cmd_cpu_sift = f"colmap feature_extractor --database_path {DB} --image_path {IMG} --ImageReader.single_camera 1 --SiftExtraction.use_gpu 0 --SiftExtraction.num_threads 4"
run_cmd("Feature Extraction", cmd_gpu_sift, cmd_cpu_sift)

# 2. Sequential matching: GPU matching with auto CPU fallback
cmd_gpu_match = f"colmap sequential_matcher --database_path {DB} --SiftMatching.use_gpu 1"
cmd_cpu_match = f"colmap sequential_matcher --database_path {DB} --SiftMatching.use_gpu 0 --SiftMatching.num_threads 4"
run_cmd("Sequential Feature Matching", cmd_gpu_match, cmd_cpu_match)

# 3. Sparse Reconstruction (Mapper)
cmd_mapper = f"colmap mapper --database_path {DB} --image_path {IMG} --output_path {SPARSE}"
run_cmd("Sparse Reconstruction (Mapper)", cmd_mapper)

# Locate the generated model directory (typically sparse/0)
model_dirs = [os.path.join(SPARSE, d) for d in os.listdir(SPARSE) if os.path.isdir(os.path.join(SPARSE, d))]
if not model_dirs:
    raise RuntimeError("COLMAP Mapper did not produce any sparse model. Check extracted image sharpness.")
sparse_0 = model_dirs[0]

# 4. Image Undistortion
cmd_undistort = f"colmap image_undistorter --image_path {IMG} --input_path {sparse_0} --output_path {DENSE} --output_type COLMAP"
run_cmd("Image Undistortion", cmd_undistort)

# 5. Dense Stereo Reconstruction (CUDA PatchMatch with Model Converter fallback)
cmd_patchmatch = f"xvfb-run -a colmap patch_match_stereo --workspace_path {DENSE} --workspace_format COLMAP --PatchMatchStereo.geom_consistency true --PatchMatchStereo.gpu_index 0"
patchmatch_ok = run_cmd("Dense Stereo (PatchMatch CUDA)", cmd_patchmatch)

fused_path = f"{DENSE}/fused.ply"
if patchmatch_ok:
    cmd_fusion = f"colmap stereo_fusion --workspace_path {DENSE} --workspace_format COLMAP --input_type geometric --output_path {fused_path}"
    run_cmd("Stereo Fusion", cmd_fusion)

# Fallback: if fused.ply was not produced, convert sparse model directly to PLY
if not os.path.exists(fused_path) or os.path.getsize(fused_path) == 0:
    print("\n⚠️ Note: Dense stereo unavailable; converting sparse point cloud to PLY...")
    cmd_convert = f"colmap model_converter --input_path {sparse_0} --output_path {fused_path} --output_type PLY"
    os.system(cmd_convert)

if not os.path.exists(fused_path):
    raise FileNotFoundError(f"Failed to produce point cloud at {fused_path}")

fused_size = os.path.getsize(fused_path) / (1024*1024)
print(f"\n✅ Point cloud ready: {fused_path} ({fused_size:.1f} MB)")

# --- Screened Poisson Surface Reconstruction ---
print(f"\n{'='*55}\n  ▶ Step 7/8: Screened Poisson Meshing (PyMeshLab)\n{'='*55}")

import pymeshlab

ms = pymeshlab.MeshSet()
ms.load_new_mesh(fused_path)
print(f"   Loaded point cloud: {ms.current_mesh().vertex_number()} vertices")

# Compute normals for point set
ms.compute_normal_for_point_sets(k=20, smoothiter=3, flipflag=True)
print('   ✅ Point normals computed (k=20, smoothiter=3)')

# Screened Poisson Surface Reconstruction
ms.generate_surface_reconstruction_screened_poisson(depth=9, preclean=True, scale=1.0)
print(f'   ✅ Poisson surface generated: {ms.current_mesh().vertex_number()} vertices, {ms.current_mesh().face_number()} faces')

# Trim low-confidence Poisson extrapolation using quality scalar
try:
    ms.compute_selection_by_condition_per_vertex(condselect='(q < 0.5)')
    ms.meshing_remove_selected_vertices_and_faces()
    print(f'   ✅ Trimmed low-confidence regions: {ms.current_mesh().face_number()} faces remaining')
except Exception as e:
    print(f'   ℹ️ Quality trimming skipped: {e}')

# Remove small disconnected noise components
try:
    ms.meshing_remove_connected_component_by_face_number(mincomponentsize=100)
    print(f'   ✅ Removed small components: {ms.current_mesh().face_number()} faces remaining')
except Exception as e:
    print(f'   ℹ️ Component cleanup skipped: {e}')

# Decimate to 150K faces for optimal 60 FPS web rendering
TARGET_FACES = 150000
if ms.current_mesh().face_number() > TARGET_FACES:
    print(f"\n{'='*55}\n  ▶ Step 8/8: Quadric Decimation (Budget: {TARGET_FACES} faces)\n{'='*55}")
    ms.meshing_decimation_quadric_edge_collapse(targetfacenum=TARGET_FACES, preservenormal=True)
    print(f"   ✅ Decimated to {ms.current_mesh().face_number()} faces")

# Transfer vertex colors from original point cloud if available
try:
    if ms.mesh_number() > 1:
        ms.set_current_mesh(1)
        ms.transfer_attributes_per_vertex(sourcemesh=0, targetmesh=1)
        print("   ✅ Vertex colors transferred")
except Exception as e:
    print(f"   ℹ️ Color transfer notice: {e}")

ms.save_current_mesh('workspace/room.ply')

total_time = time.time() - start
print(f"\n🎉 3D Reconstruction complete! Total elapsed: {total_time:.0f}s ({total_time/60:.1f} min)")


## 4. GLB Export & Download
Converts the vertex-colored mesh to binary `.glb` format for the Three.js web viewer.

In [ ]:
#@title 4️⃣ GLB Export & Download
import trimesh

mesh = trimesh.load('workspace/room.ply', force='mesh')
print(f"📐 Mesh loaded: {len(mesh.vertices)} vertices, {len(mesh.faces)} faces")

# Check for vertex colors
if hasattr(mesh.visual, 'vertex_colors') and mesh.visual.vertex_colors is not None:
    print(f"🎨 Vertex colors: {mesh.visual.vertex_colors.shape}")
else:
    print("⚠️ No vertex colors found — model will appear gray in viewer")

# Export as binary GLB
glb_data = mesh.export(file_type='glb')
with open('room.glb', 'wb') as f:
    f.write(glb_data)

size_mb = os.path.getsize('room.glb') / (1024*1024)
print(f"\n✅ Exported: room.glb ({size_mb:.1f} MB)")

# Auto-download
from google.colab import files
files.download('room.glb')

print("\n🎉 Done! Place room.glb in public/models/ of your web viewer.")
print("   Or use the 'Load .GLB Model' button in the viewer to load it directly.")